# Week 3 · Part 3 — Diagnose, improve, verify, report once

**Question for today:** Part 2 showed that the six-input model makes errors that depend on distance. Can we fix that, and **prove** the fix works?

We follow the cycle that good model building uses:

**diagnose → change → verify with cross-validation → check the residuals again → test once → report**

### Plan
1. Setup: the same data, split, folds, and helpers as Part 2
2. Recall the problem
3. Why a straight line in metres bends
4. Build the log-distance model
5. Verify the change with cross-validation
6. Did the residual pattern shrink?
7. Residuals as a probability distribution
8. Decide, then open the test rows once
9. Read the log coefficient
10. Extension: repeated locations
11. Evidence statement

### Words you will meet
| Word | Meaning |
|---|---|
| transformation | replacing an input by a function of it, such as its logarithm |
| log scale | a scale on which equal steps are equal **ratios** |
| normal distribution | the bell-shaped distribution assumed for errors in Lecture 3 |
| test once | the test rows are used a single time, after every decision is made |

## 1. Setup
The same tools, data, split, and folds as Part 2. The test rows stay locked until section 8.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer          # choose or transform input columns
from sklearn.dummy import DummyRegressor               # baseline: always predict the training mean
from sklearn.linear_model import LinearRegression      # least-squares regression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline                  # chain preprocessing and model
from sklearn.preprocessing import FunctionTransformer, StandardScaler

In [ ]:
df = pd.read_csv("Dataset.csv")
target = "price_10k_twd_per_ping"
features = ["transaction_date_decimal", "house_age_years", "distance_to_mrt_m",
            "convenience_stores", "latitude", "longitude"]

In [ ]:
train, test = train_test_split(df, test_size=0.20, random_state=42)
X_train, y_train = train[features], train[target]
cv = KFold(n_splits=5, shuffle=True, random_state=42)
print("training rows:", len(train), "  locked test rows:", len(test))

The same helpers as in Part 2: `uses` picks input columns, `scoring` lists the three metrics, and `evaluate` cross-validates a dictionary of models and returns a summary table and a fold table.

In [ ]:
def uses(columns):
    """Pass only the named input columns to the model."""
    return ColumnTransformer([("keep", "passthrough", columns)])

scoring = {"MAE": "neg_mean_absolute_error",
           "RMSE": "neg_root_mean_squared_error",
           "R2": "r2"}

def evaluate(models):
    """Cross-validate every model on the same five training folds."""
    rows, fold_mae = [], {}
    for name, model in models.items():
        result = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
        mae = -result["test_MAE"]
        fold_mae[name] = mae
        rows.append({"model": name,
                     "MAE": mae.mean(),
                     "MAE fold SD": mae.std(ddof=1),
                     "RMSE": -result["test_RMSE"].mean(),
                     "R2": result["test_R2"].mean()})
    table = pd.DataFrame(rows).set_index("model").round(3)
    folds = pd.DataFrame(fold_mae, index=[f"fold {i}" for i in range(1, 6)]).T.round(2)
    return table, folds


## 2. Recall the problem
Part 2 ended with the out-of-fold residuals of the six-input model, averaged in six distance bands. We compute them again, this time with a small function, because we will need the same check for a second model later.

`residual_bands(model)` makes out-of-fold predictions, computes residuals, and averages them in the same six distance bands as Part 2.

In [ ]:
distance_edges = [0, 300, 600, 1000, 2000, 3000, 7000]   # band boundaries in metres

def residual_bands(model):
    predicted = cross_val_predict(model, X_train, y_train, cv=cv)
    residual = y_train - predicted
    band = pd.cut(X_train["distance_to_mrt_m"], distance_edges)
    return residual.groupby(band, observed=True).mean(), predicted

six_inputs = Pipeline([("inputs", uses(features)), ("model", LinearRegression())])
before, oof_before = residual_bands(six_inputs)
before.round(2).to_frame("mean residual")

**What we see:** the U from Part 2: +4.72 up to 300 m, −6.71 and −5.31 between 1 and 3 km, +6.18 beyond 3 km. Too low near and far, too high in the middle. The model treats the effect of distance as a straight line, but it is not.

## 3. Why a straight line in metres bends
Look at how the distances are spread:

In [ ]:
X_train["distance_to_mrt_m"].describe().round(1)

**What we see:** half of the sales are within 492 m of a station, three quarters within about 1.4 km, but the maximum is 6.3 km. The distribution has a **long right tail**.

On the metre scale a straight line treats every 100 m the same. But going from 100 m to 200 m (**twice** as far) probably matters more for price than going from 3,000 m to 3,100 m (3% farther).

A **logarithm** measures ratios instead of differences: each doubling adds the same amount. We use `np.log1p(d)`, which is log(1 + d); the "+1" only protects against log(0). The table shows what it does to some distances:

In [ ]:
examples = pd.DataFrame({"distance (m)": [100, 200, 500, 1000, 3000, 3100, 6000]})
examples["log(1 + distance)"] = np.log1p(examples["distance (m)"]).round(3)
examples

**What we see:** 100 → 200 m moves the log value from 4.62 to 5.30 (+0.69), while 3,000 → 3,100 m moves it only from 8.01 to 8.04 (+0.03). On the log scale, differences near stations are stretched and differences far away are compressed.

Now the same sales on both scales:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(X_train["distance_to_mrt_m"], y_train, alpha=.45)
axes[0].set(title="Original scale", xlabel="distance (metres)", ylabel="price (10,000 TWD per Ping)")
axes[1].scatter(np.log1p(X_train["distance_to_mrt_m"]), y_train, alpha=.45)
axes[1].set(title="Log scale", xlabel="log(1 + distance)")
plt.show()

**What we see:** on the left the cloud falls steeply and then flattens: a curve. On the right it falls much more evenly, much closer to a straight line. So a straight line in **log distance** should fit better than a straight line in metres.

## 4. Build the log-distance model
**New tool — `FunctionTransformer(np.log1p)`.** It turns any function into a preprocessing step, just like the imputers and scalers in Week 2. Here it replaces each distance by log(1 + distance).

We put it in a `ColumnTransformer` with two branches:
- `"log"`: apply `log1p` to `distance_to_mrt_m`;
- `"keep"`: pass the other five inputs through unchanged.

In [ ]:
log_distance = ColumnTransformer([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"), ["distance_to_mrt_m"]),
    ("keep", "passthrough", [f for f in features if f != "distance_to_mrt_m"]),
])
log_model = Pipeline([("inputs", log_distance), ("model", LinearRegression())])

What does the model actually receive? Transform the first three training rows and put the original distance beside the result:

In [ ]:
preview = pd.DataFrame(log_distance.fit_transform(X_train.head(3)),
                       columns=log_distance.get_feature_names_out())
preview.insert(0, "original distance (m)", X_train["distance_to_mrt_m"].head(3).to_numpy())
preview.round(3)

**What we see:** the first sale, 57.6 m from a station, now enters the model as 4.07; the second, 2,216.6 m away, as 7.70. The other five columns are unchanged. The column names show which branch produced each column: `log__` or `keep__`.

**Is this still linear regression?** Yes. "Linear" refers to the **coefficients**: the prediction is still a weighted sum, $b_0 + b_1 \cdot \log(1+\text{distance}) + b_2 \cdot \text{age} + \dots$. Only one input has been transformed. Lecture 3 (slide 13) shows polynomial regression, which works the same way.

Because the transformation is inside the pipeline, every fold applies it in the same way, and new data will be transformed automatically.

## 5. Verify the change with cross-validation
Same folds, same metrics, three models. Before running: **will log distance lower the MAE, and by more than the fold spread?**

In [ ]:
models = {
    "mean dummy": DummyRegressor(strategy="mean"),
    "all six inputs": six_inputs,
    "log distance": log_model,
}
table, folds = evaluate(models)
table

In [ ]:
folds

In [ ]:
gain = folds.loc["all six inputs"] - folds.loc["log distance"]
print("MAE gain of log distance, per fold:", gain.round(2).tolist())
print("folds won:", int((gain > 0).sum()), "of", len(gain), "  mean gain:", round(gain.mean(), 3))

**What we see:**
- CV MAE falls from 6.545 to 5.555, and R² rises from 0.55 to 0.62.
- Log distance wins **all five folds**, by 0.53 to 1.45.
- The mean gain, 0.99, is about as large as the fold-to-fold SD of either model (0.87–0.94), and it appears in every fold.

This is strong evidence. But we changed the model **because of a residual pattern**, so we must also check that the pattern became smaller.

## 6. Did the residual pattern shrink?

In [ ]:
after, oof_after = residual_bands(log_model)
comparison = pd.DataFrame({"all six inputs": before, "log distance": after})
display(comparison.round(2))
print("average size of band means:", comparison.abs().mean().round(2).to_dict())

In [ ]:
print("lowest out-of-fold prediction, all six inputs:", round(oof_before.min(), 2))
print("lowest out-of-fold prediction, log distance:  ", round(oof_after.min(), 2))

**What we see:**
- Five of the six band means moved towards zero. The worst band improved from −6.71 to −3.92, and the far band from +6.18 to +3.37. Only the 600–1,000 m band, which was already small, moved slightly away (−0.49 to +1.06).
- The average size of the band means fell from 4.13 to 1.84.
- A U-shape is still visible (−3.92 at 1–2 km, +3.37 beyond 3 km). The pattern is **reduced, not removed**.
- The lowest prediction rose from 4.22 to 10.57. The straight line in metres drove far-away predictions towards zero; the log version does not.

## 7. Residuals as a probability distribution
Lecture 3 (slides 15–16) writes the model as

$$Y = \beta_0 + \beta_1 X + \varepsilon, \qquad \varepsilon \sim N(0, \sigma^2)$$

The error ε is assumed to follow a **normal distribution**: bell-shaped, centred at 0, with standard deviation σ. For a normal distribution about **68%** of values lie within one σ of the centre and about **95%** within two.

Under this assumption, least squares gives the most likely coefficients (the *maximum-likelihood* estimate). The residuals are our only evidence about ε: where they are centred, how wide they are, and how heavy their tails are.

In [ ]:
residual = y_train - oof_after
mean, sd = residual.mean(), residual.std(ddof=1)
print("residual mean:", round(mean, 3), "  residual SD:", round(sd, 3))
print("share within 1 SD:", round((residual.abs() <= sd).mean(), 3), "(normal: 0.683)")
print("share within 2 SD:", round((residual.abs() <= 2 * sd).mean(), 3), "(normal: 0.954)")

The histogram compares the residuals with a normal curve that has the same mean and SD:

In [ ]:
grid = np.linspace(residual.min(), residual.max(), 300)
normal_curve = np.exp(-(grid - mean) ** 2 / (2 * sd ** 2)) / (sd * np.sqrt(2 * np.pi))
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(residual, bins=30, density=True, edgecolor="white", label="out-of-fold residuals")
ax.plot(grid, normal_curve, color="#D1495B", linewidth=2.5, label="normal curve, same mean and SD")
ax.set(title="Residual distribution, log-distance model", xlabel="actual − predicted", ylabel="density")
ax.legend(); plt.show()

**What we see:**
- The mean is almost exactly 0, as the model assumes.
- 84% of residuals lie within one SD, much more than the 68% of a normal curve. The histogram is **taller and narrower** in the middle than the bell.
- 95.5% lie within two SD, as for a normal curve, but a few residuals are far out on the **right**: some sales are much more expensive than predicted.

So the normal assumption is only roughly true. Those far-out residuals matter because least squares squares them, so they pull the line towards themselves. Which sales are they?

In [ ]:
largest = train.loc[residual.abs().sort_values(ascending=False).index[:5],
                    ["record_id", target, "distance_to_mrt_m", "house_age_years", "convenience_stores"]]
largest["predicted"] = pd.Series(oof_after, index=train.index)[largest.index]
largest.round(2)

**What we see:**
- Record 271 sold for 117.5, while the model predicted 44.7. It is 253 m from a station and has one store nearby; nothing in its inputs explains such a price. It might be an unusual property or a recording error, and the dataset cannot tell us which.
- Record 114 is the opposite case: the cheapest sale in the data (7.6), predicted at 41.4.
- Four of the five largest misses are **underpredictions**, which matches the long right tail in the histogram.

**Do not delete it just because its residual is large.** Deleting awkward rows makes a model look better than it is. Investigate, report it, and state it as a limitation.

## 8. Decide, then open the test rows once
The decision uses **only cross-validation**:

| Check | Result |
|---|---|
| Size of gain | 0.99 MAE, about one fold SD |
| Consistency | better in 5 of 5 folds |
| Reason | it reduced the residual pattern that motivated it |

We choose the log-distance model. Now, and only now, we fit it on **all 331 training rows** and evaluate it **once** on the 83 locked test rows, with the dummy as the reference.

In [ ]:
X_test, y_test = test[features], test[target]
final = log_model.fit(X_train, y_train)
reference = DummyRegressor(strategy="mean").fit(X_train, y_train)

rows = []
for name, model in [("mean dummy", reference), ("log distance (final)", final)]:
    predicted = model.predict(X_test)
    rows.append({"model": name,
                 "test MAE": mean_absolute_error(y_test, predicted),
                 "test RMSE": root_mean_squared_error(y_test, predicted),
                 "test R2": r2_score(y_test, predicted)})
pd.DataFrame(rows).set_index("model").round(3)

**What we see:** on the test rows the final model misses by 4.39 on average (dummy 10.92), with R² 0.73.

The test MAE is a little better than the cross-validation mean (5.56), and even slightly better than the best single fold. The 83 test rows are also **one draw**. Report both numbers: cross-validation estimates typical performance, and the test result checks it once on untouched rows.

**From here the model is fixed.** If we now tried other models and kept whichever did best on these rows, the test rows would become just another validation set, and we would have no honest estimate left.

## 9. Read the log coefficient

In [ ]:
coefficients = pd.Series(final.named_steps["model"].coef_,
                         index=final.named_steps["inputs"].get_feature_names_out())
coefficients.round(4).to_frame("coefficient")

The log-distance coefficient, −7.10, is per one unit of log distance, which is hard to imagine. A more natural question is: **what if distance doubles?** Doubling adds log(2) ≈ 0.693 to the log value, so we multiply:

In [ ]:
b = coefficients["log__distance_to_mrt_m"]
print("fitted change when distance doubles:", round(b * np.log(2), 2), "price units")

**What we see:** doubling the distance to the station is associated with a predicted price about 4.9 units lower, holding the other inputs fixed. Because the model uses log(1 + distance) and all distances are above 20 m, this is a very close approximation.

The same warning as before: this is an association in this sample, not the effect of moving a flat.

## 10. Extension: repeated locations
Many sales share **exactly the same coordinates**, and the data have no building identifier. If sales from the same place fall both in a fitting fold and in a scoring fold, the model may be rewarded for "recognising" a place rather than for predicting a new one. How many test rows are at places the model has already seen?

In [ ]:
train_places = set(zip(train["latitude"], train["longitude"]))
shared = sum(place in train_places for place in zip(test["latitude"], test["longitude"]))
print("test rows at a location that also appears in training:", shared, "of", len(test))

**New tool — `GroupKFold`.** It builds folds so that all rows of one **group** (here, one location) stay together in the same fold. Then no location is ever both fitted and scored.

In [ ]:
from sklearn.model_selection import GroupKFold
location = train.groupby(["latitude", "longitude"]).ngroup()
grouped = cross_validate(log_model, X_train, y_train, cv=GroupKFold(n_splits=5),
                         groups=location, scoring="neg_mean_absolute_error")
print("CV MAE, locations kept together:", round(-grouped["test_score"].mean(), 3))
print("CV MAE, ordinary 5-fold:        ", table.loc["log distance", "MAE"])

**What we see:** 44 of 83 test rows share a location with training rows. But keeping locations together barely changes the estimate (5.50 against 5.56). Our conclusion does not depend on repeated locations; we still state them as a limitation.

## 11. Evidence statement
A good conclusion says **how** the model was chosen, gives the numbers **with units**, compares with the baseline, and says **where the claim stops**:

> On this dataset, linear regression with distance on a log scale was chosen by five-fold cross-validation on 331 training rows: MAE 5.56 against 6.55 for the untransformed six-input model, better in all five folds. Evaluated once on 83 locked test rows it reached MAE 4.39 price units (10,000 TWD per Ping) and R² 0.73, against MAE 10.92 for the mean dummy. The data are a small historical sample from one district, omit property characteristics such as floor area and condition, and include repeated locations without property identifiers. Coefficients describe associations in this sample, not effects of changing a property.

### What you learned
- A residual pattern suggests a specific change; here, a log scale for distance.
- A transformed input keeps the model linear in its coefficients.
- A change is accepted on validation evidence: size, consistency across folds, and a smaller residual pattern.
- Residuals can be compared with the normal distribution assumed in the lecture; here the tails are heavier.
- The test rows are used once, after the decision.

In **Practice 3B** you run this cycle yourself, with a change of your own.

**Sources:** AITU Lecture 3, slides 13 and 15–16; ISLP, physical PDF pages 109–110 (residual plots and predictor transformations); Bishop, *Pattern Recognition and Machine Learning*, physical PDF page 158 (models linear in their parameters); Ng and Ma, *CS229 Lecture Notes*, physical PDF pages 18–19 (least squares as maximum likelihood); UCI DOI 10.24432/C5J30W.